In [1]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import eu9_osm
from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation, PillowWriter

In [2]:
place="Lisbon, Portugal"
gdf = eu9_osm.features_from_place(place, tags={"highway": True})
gdf = gdf.to_crs("EPSG:3763")

In [3]:
# Missing street names 
missing = (gdf["name"].isna() | (gdf["name"].astype("string").str.strip() == "").fillna(False)).to_numpy(dtype=bool)

In [4]:
# Classify streets
named_lines = []
null_lines = []

for geom, is_null in zip(gdf.geometry, missing):

    if geom is None or geom.is_empty:
        continue
    if geom.geom_type == "LineString":
        lines = [geom]
    elif geom.geom_type == "MultiLineString":
        lines = geom.geoms
    else:
        continue

    for line in lines:
        coords = np.asarray(line.coords)[:, :2]
        if is_null:
            null_lines.append(coords)
        else:
            named_lines.append(coords)

In [5]:
# Random fading
rng = np.random.default_rng(42)
fade_start = rng.uniform(0, 0.8, len(null_lines))
fade_duration = 0.2

In [7]:
fig, ax = plt.subplots(figsize=(8, 8), facecolor="#111318")

# Named streets 
named_collection = LineCollection(named_lines, colors="#F1EDE3", lw=0.45)
ax.add_collection(named_collection)

# Unnamed streets 
null_collection = LineCollection(null_lines, colors="#F1EDE3", lw=0.45)
ax.add_collection(null_collection)

xmin, ymin, xmax, ymax = gdf.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal")
ax.axis("off")

title = ax.set_title("No name, no street?", y=1.05, size=25, weight="bold", color="#F1EDE3", pad=20)

counter = ax.text(0.5, 1.02, "", color="#A7A7A7", size=15, transform=ax.transAxes, ha="center")

fig.text(0.5, 0.05,
        "Data: OpenStreetMap (licensed under ODbL) | Map: Bernadett Piros",
         size=8, color="#F1EDE3", ha="center")

# Update
FRAMES = 100
FRAME_DURATION = 100

# RGBA array per null line
colors = np.tile([241/255, 237/255, 227/255, 1.0], (len(null_lines), 1))

def update(frame):

    progress = frame / (FRAMES - 1)

    # Opacity change
    alpha = 1 - np.clip((progress - fade_start) / fade_duration, 0, 1)

    colors[:, 3] = alpha

    null_collection.set_color(colors)

    counter.set_text(
        f"Unnamed streets fading: {progress:.0%}"
    )

    return null_collection, counter

# Animation
animation = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=FRAME_DURATION,
    blit=True
)

animation.save("unnamed.gif", writer=PillowWriter(fps=1000 / FRAME_DURATION), dpi=120)

plt.close(fig)